In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

# -----------------------------
# 1. Load Dataset (robust way)
# -----------------------------
data = pd.read_csv(
    "IMDB_Movie_Review.csv",
    encoding='latin-1',     # fixes Unicode error
    engine='python',        # handles messy CSV
    on_bad_lines='skip'     # skips corrupted rows
)

# Verify structure
print("Shape:", data.shape)
print("Columns:", data.columns)

# -----------------------------
# 2. Basic Cleaning
# -----------------------------
# Drop missing rows
data = data.dropna()

# Ensure correct columns exist
if 'review' not in data.columns or 'sentiment' not in data.columns:
    raise ValueError("Dataset must contain 'review' and 'sentiment' columns")

# Convert sentiment to numeric
data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})

# Remove rows where mapping failed
data = data.dropna(subset=['sentiment'])

# -----------------------------
# 3. Text Preprocessing
# -----------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

data['review'] = data['review'].apply(clean_text)

# -----------------------------
# 4. Feature Extraction
# -----------------------------
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(data['review']).toarray()
y = data['sentiment'].astype(int)

# -----------------------------
# 5. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 6. Model Definition
# -----------------------------
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # better than hardcoding 5000
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# 7. Training (with early stopping)
# -----------------------------
early_stop = EarlyStopping(
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# -----------------------------
# 8. Evaluation
# -----------------------------
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("\nFinal Test Accuracy:", round(acc, 4))

# -----------------------------
# 9. Sample Predictions
# -----------------------------
pred = model.predict(X_test[:5])

print("\nSample Predictions (Actual | Predicted):")
for i in range(5):
    print(y_test.iloc[i], "|", round(pred[i][0], 3))

Shape: (49613, 2)
Columns: Index(['review', 'sentiment'], dtype='object')
Epoch 1/10
993/993 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8557 - loss: 0.3545 - val_accuracy: 0.8942 - val_loss: 0.2748
Epoch 2/10
993/993 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8990 - loss: 0.2608 - val_accuracy: 0.8854 - val_loss: 0.2985
Epoch 3/10
993/993 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9083 - loss: 0.2351 - val_accuracy: 0.8890 - val_loss: 0.2865

Final Test Accuracy: 0.8893
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step

Sample Predictions (Actual | Predicted):
0 | 0.869
0 | 0.627
0 | 0.014
1 | 0.992
0 | 0.653


In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping



# =========================
# 2. LOAD DATASET
# =========================
df = pd.read_csv(
    "IMDB_Dataset.csv",
    encoding_errors='ignore',
    on_bad_lines='skip',
    engine='python'
)

print("Dataset Loaded ✅")



# =========================
# 3. DATA CLEANING
# =========================
df = df.dropna()
df.columns = ['review', 'sentiment']



# =========================
# 4. TEXT PREPROCESSING
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)        # Remove HTML tags
    text = re.sub(r'[^a-zA-Z ]', '', text)   # Remove special chars
    return text

df['review'] = df['review'].apply(clean_text)




# =========================
# 5. LABEL ENCODING
# =========================
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("\nSentiment Count:")
print(df['sentiment'].value_counts())




# =========================
# 6. TF-IDF VECTORIZATION
# =========================
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review']).toarray()
y = df['sentiment']





# =========================
# 7. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)




# =========================
# 8. BUILD DNN MODEL
# =========================
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    
    Dense(128, activation='relu'),
    Dropout(0.5),
    
    Dense(64, activation='relu'),
    Dropout(0.5),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()






# =========================
# 9. EARLY STOPPING
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)






# =========================
# 10. TRAIN MODEL
# =========================
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)





# =========================
# 11. PLOT GRAPH
# =========================
plt.figure()

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Accuracy Graph")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.show()
